In [2]:
# ================================================================
# CONTRACT CLAUSE DATASET -> EMBEDDINGS -> QDRANT CLOUD
# Google Colab | Single Cell
# ================================================================

# ================================================================
# 0. INSTALL REQUIRED PACKAGES
# ================================================================

print("\n" + "=" * 70)
print("STEP 0: INSTALLING DEPENDENCIES")
print("=" * 70)

!pip -q install -U pandas sentence-transformers qdrant-client

print("✅ Dependencies installed successfully")


# ================================================================
# 1. IMPORTS
# ================================================================

print("\n" + "=" * 70)
print("STEP 1: IMPORTING LIBRARIES")
print("=" * 70)

import os
import time
import pandas as pd

from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct
)

print("✅ Libraries imported successfully")


# ================================================================
# 2. UPLOAD CSV
# ================================================================

print("\n" + "=" * 70)
print("STEP 2: UPLOAD CSV DATASET")
print("=" * 70)

from google.colab import files

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("❌ No CSV file was uploaded.")

# Get uploaded filename
csv_file = list(uploaded.keys())[0]

print(f"✅ File uploaded: {csv_file}")


# ================================================================
# 3. LOAD DATASET
# ================================================================

print("\n" + "=" * 70)
print("STEP 3: LOADING DATASET")
print("=" * 70)

df = pd.read_csv(csv_file)

print(f"✅ Dataset loaded")
print(f"📊 Rows    : {len(df)}")
print(f"📊 Columns : {list(df.columns)}")


# ================================================================
# 4. VALIDATE REQUIRED COLUMNS
# ================================================================

print("\n" + "=" * 70)
print("STEP 4: VALIDATING DATASET STRUCTURE")
print("=" * 70)

required_columns = [
    "filename",
    "clause_type",
    "clause_text"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"❌ Missing required columns: {missing_columns}"
    )

print("✅ Required columns found:")
for col in required_columns:
    print(f"   ✓ {col}")


# ================================================================
# 5. CLEAN DATA
# ================================================================

print("\n" + "=" * 70)
print("STEP 5: CLEANING DATA")
print("=" * 70)

before_count = len(df)

# Remove rows where clause text is missing
df = df.dropna(subset=["clause_text"])

# Convert clause text to string
df["clause_text"] = (
    df["clause_text"]
    .astype(str)
    .str.strip()
)

# Remove empty clauses
df = df[df["clause_text"] != ""]

# Remove duplicate clause text
df = df.drop_duplicates(
    subset=["clause_text"]
).reset_index(drop=True)

after_count = len(df)

print(f"Original rows       : {before_count}")
print(f"Rows after cleaning : {after_count}")
print(f"Rows removed        : {before_count - after_count}")

print("✅ Data cleaning completed")


# ================================================================
# 6. SHOW DATASET SAMPLE
# ================================================================

print("\n" + "=" * 70)
print("STEP 6: DATASET SAMPLE")
print("=" * 70)

print(
    df[
        ["filename", "clause_type", "clause_text"]
    ].head(3).to_string(index=False)
)


# ================================================================
# 7. LOAD EMBEDDING MODEL
# ================================================================

print("\n" + "=" * 70)
print("STEP 7: LOADING EMBEDDING MODEL")
print("=" * 70)

MODEL_NAME = "all-MiniLM-L6-v2"

print(f"Model: {MODEL_NAME}")
print("Loading model...")

model = SentenceTransformer(MODEL_NAME)

# Determine vector dimension automatically
VECTOR_SIZE = model.get_sentence_embedding_dimension()

print(f"✅ Model loaded successfully")
print(f"📐 Vector dimension: {VECTOR_SIZE}")


# ================================================================
# 8. GENERATE EMBEDDINGS
# ================================================================

print("\n" + "=" * 70)
print("STEP 8: GENERATING EMBEDDINGS")
print("=" * 70)

texts = df["clause_text"].tolist()

print(f"Total clauses to embed: {len(texts)}")
print("Generating embeddings...")
print("This may take some time.\n")

start_time = time.time()

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

embedding_time = time.time() - start_time

print("\n✅ Embedding generation completed")
print(f"⏱️ Time taken: {embedding_time:.2f} seconds")
print(f"📐 Embedding shape: {embeddings.shape}")

# Safety check
if embeddings.shape[0] != len(df):
    raise RuntimeError(
        "❌ Number of embeddings does not match number of rows."
    )

if embeddings.shape[1] != VECTOR_SIZE:
    raise RuntimeError(
        "❌ Embedding dimension mismatch."
    )

print("✓ Number of vectors matches number of clauses")
print("✓ Vector dimensions are correct")


# ================================================================
# 9. QDRANT CREDENTIALS
# ================================================================

print("\n" + "=" * 70)
print("STEP 9: QDRANT CLOUD CONFIGURATION")
print("=" * 70)

# ------------------------------------------------
# IMPORTANT:
#
# Replace these with your Qdrant Cloud credentials.
#
# For a shared/public notebook, DO NOT hard-code
# your API key.
#
# Better option: use Colab Secrets.
# ------------------------------------------------

QDRANT_URL =""
QDRANT_API_KEY= ""

if (
    QDRANT_URL == "YOUR_QDRANT_URL"
    or QDRANT_API_KEY == "YOUR_QDRANT_API_KEY"
):
    raise ValueError(
        """
        ❌ Qdrant credentials are not configured.

        Set:

        QDRANT_URL = "https://....qdrant.io"
        QDRANT_API_KEY = "..."

        Then run the cell again.
        """
    )

print("✓ Qdrant URL configured")
print("✓ Qdrant API key configured")


# ================================================================
# 10. CONNECT TO QDRANT
# ================================================================

print("\n" + "=" * 70)
print("STEP 10: CONNECTING TO QDRANT CLOUD")
print("=" * 70)

client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY
)

print("Testing Qdrant connection...")

try:
    collections = client.get_collections()
    print("✅ Successfully connected to Qdrant Cloud")
    print(
        f"Existing collections: "
        f"{len(collections.collections)}"
    )

except Exception as e:
    raise RuntimeError(
        f"❌ Qdrant connection failed: {e}"
    )


# ================================================================
# 11. CREATE QDRANT COLLECTION
# ================================================================

print("\n" + "=" * 70)
print("STEP 11: CREATING QDRANT COLLECTION")
print("=" * 70)

COLLECTION_NAME = "contract_clauses"

print(f"Collection name: {COLLECTION_NAME}")

# Check whether collection already exists
existing_collections = [
    collection.name
    for collection in client.get_collections().collections
]

if COLLECTION_NAME in existing_collections:

    print(
        f"⚠️ Collection '{COLLECTION_NAME}' already exists."
    )

    user_choice = input(
        "Do you want to DELETE and recreate it? (yes/no): "
    ).strip().lower()

    if user_choice == "yes":

        print("Deleting existing collection...")

        client.delete_collection(
            collection_name=COLLECTION_NAME
        )

        print("✓ Existing collection deleted")

    else:

        raise RuntimeError(
            """
            ❌ Collection already exists.

            Stop here to prevent accidental duplicate ingestion.

            If you want to recreate it, run again and type 'yes'.
            """
        )


# Create collection
client.create_collection(
    collection_name=COLLECTION_NAME,

    vectors_config=VectorParams(
        size=VECTOR_SIZE,
        distance=Distance.COSINE
    )
)

print("✅ Collection created")
print(f"📐 Vector size: {VECTOR_SIZE}")
print("📏 Distance: COSINE")


# ================================================================
# 12. PREPARE QDRANT POINTS
# ================================================================

print("\n" + "=" * 70)
print("STEP 12: PREPARING VECTORS + PAYLOAD")
print("=" * 70)

"""
Each Qdrant point contains THREE important components:

    ID
    │
    ├── Vector
    │      └── 384-dimensional embedding
    │
    └── Payload
           ├── filename
           ├── clause_type
           └── clause_text

Example:

{
    "id": 0,

    "vector": [0.01, -0.04, ...],

    "payload": {
        "filename": "contract1.pdf",
        "clause_type": "Termination",
        "clause_text": "..."
    }
}
"""

points = []

for idx, (_, row) in enumerate(df.iterrows()):

    point = PointStruct(

        # ------------------------------------------------
        # UNIQUE VECTOR ID
        # ------------------------------------------------
        id=idx,

        # ------------------------------------------------
        # EMBEDDING VECTOR
        # ------------------------------------------------
        vector=embeddings[idx].tolist(),

        # ------------------------------------------------
        # PAYLOAD / METADATA
        # ------------------------------------------------
        payload={
            "filename": str(row["filename"]),

            "clause_type": str(
                row["clause_type"]
            ),

            "clause_text": str(
                row["clause_text"]
            )
        }
    )

    points.append(point)

print(f"✅ Prepared {len(points)} Qdrant points")


# ================================================================
# 13. UPLOAD TO QDRANT IN BATCHES
# ================================================================

print("\n" + "=" * 70)
print("STEP 13: UPLOADING VECTORS TO QDRANT")
print("=" * 70)

BATCH_SIZE = 256

total_points = len(points)

print(f"Total vectors : {total_points}")
print(f"Batch size    : {BATCH_SIZE}")

start_time = time.time()

for start in range(
    0,
    total_points,
    BATCH_SIZE
):

    end = min(
        start + BATCH_SIZE,
        total_points
    )

    batch = points[start:end]

    client.upsert(
        collection_name=COLLECTION_NAME,
        points=batch,
        wait=True
    )

    completed = end

    percentage = (
        completed / total_points
    ) * 100

    print(
        f"✅ Uploaded "
        f"{completed}/{total_points} "
        f"({percentage:.1f}%)"
    )

upload_time = time.time() - start_time

print("\n✅ ALL VECTORS UPLOADED")
print(f"⏱️ Upload time: {upload_time:.2f} seconds")


# ================================================================
# 14. VERIFY QDRANT COLLECTION
# ================================================================

print("\n" + "=" * 70)
print("STEP 14: VERIFYING QDRANT")
print("=" * 70)

collection_info = client.get_collection(
    collection_name=COLLECTION_NAME
)

print("✅ Collection verification completed")

print(
    f"📊 Points stored: "
    f"{collection_info.points_count}"
)

print(
    f"📐 Vector size: "
    f"{VECTOR_SIZE}"
)

print("📏 Distance: COSINE")


if collection_info.points_count != len(df):

    print(
        "⚠️ WARNING: Point count does not match "
        "number of dataset rows."
    )

else:

    print(
        "✅ Point count matches dataset rows"
    )


# ================================================================
# 15. TEST SEMANTIC SEARCH
# ================================================================

print("\n" + "=" * 70)
print("STEP 15: TESTING SEMANTIC SEARCH")
print("=" * 70)

test_query = """
What happens if the company undergoes a change of control?
"""

print(f"🔎 Query:\n{test_query}")

# Convert question into same embedding space
query_vector = model.encode(
    test_query,
    normalize_embeddings=True
).tolist()

print("✓ Query converted into embedding")

# Search Qdrant
search_results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=5
).points

print(
    f"✓ Retrieved {len(search_results)} results"
)

print("\n" + "-" * 70)
print("TOP 5 SIMILAR CLAUSES")
print("-" * 70)

for rank, result in enumerate(
    search_results,
    start=1
):

    print(f"\nRESULT #{rank}")

    print(
        f"Similarity Score : "
        f"{result.score:.4f}"
    )

    print(
        f"Clause Type      : "
        f"{result.payload['clause_type']}"
    )

    print(
        f"Filename         : "
        f"{result.payload['filename']}"
    )

    print(
        f"Clause Text      :\n"
        f"{result.payload['clause_text'][:1000]}"
    )

    print("-" * 70)


# ================================================================
# 16. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 70)
print("🎉 INGESTION COMPLETED SUCCESSFULLY")
print("=" * 70)

print(f"""
Dataset
-------
Original rows       : {before_count}
Clean rows           : {after_count}

Embedding
---------
Model                : {MODEL_NAME}
Vector dimensions    : {VECTOR_SIZE}
Vectors generated    : {len(embeddings)}

Qdrant
------
Collection           : {COLLECTION_NAME}
Vectors stored       : {collection_info.points_count}
Distance metric      : COSINE

Payload
-------
✓ filename
✓ clause_type
✓ clause_text

Pipeline
--------
CSV
 ↓
Cleaning
 ↓
Embedding
 ↓
Vector + Payload
 ↓
Qdrant Cloud
 ↓
Semantic Search
 ✓

NEXT STEP:
Upload a user contract → extract its clauses → embed the
question/clauses → retrieve relevant reference clauses from
this Qdrant collection → use LangGraph + LLM + Pydantic
to generate the structured contract analysis.
""")

print("=" * 70)
print("🚀 READY FOR THE NEXT STAGE")
print("=" * 70)


STEP 0: INSTALLING DEPENDENCIES
✅ Dependencies installed successfully

STEP 1: IMPORTING LIBRARIES
✅ Libraries imported successfully

STEP 2: UPLOAD CSV DATASET


Saving all_reshaped_clauses.csv to all_reshaped_clauses (1).csv
✅ File uploaded: all_reshaped_clauses (1).csv

STEP 3: LOADING DATASET
✅ Dataset loaded
📊 Rows    : 8628
📊 Columns : ['filename', 'clause_type', 'clause_text']

STEP 4: VALIDATING DATASET STRUCTURE
✅ Required columns found:
   ✓ filename
   ✓ clause_type
   ✓ clause_text

STEP 5: CLEANING DATA
Original rows       : 8628
Rows after cleaning : 7231
Rows removed        : 1397
✅ Data cleaning completed

STEP 6: DATASET SAMPLE
                                                                                          filename       clause_type                                                                                                                                                                                                                                                                                                                                                                                                          

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Model loaded successfully
📐 Vector dimension: 384

STEP 8: GENERATING EMBEDDINGS
Total clauses to embed: 7231
Generating embeddings...
This may take some time.



/tmp/ipykernel_712/485182546.py:176: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  VECTOR_SIZE = model.get_sentence_embedding_dimension()


Batches:   0%|          | 0/113 [00:00<?, ?it/s]


✅ Embedding generation completed
⏱️ Time taken: 9.84 seconds
📐 Embedding shape: (7231, 384)
✓ Number of vectors matches number of clauses
✓ Vector dimensions are correct

STEP 9: QDRANT CLOUD CONFIGURATION
✓ Qdrant URL configured
✓ Qdrant API key configured

STEP 10: CONNECTING TO QDRANT CLOUD
Testing Qdrant connection...
✅ Successfully connected to Qdrant Cloud
Existing collections: 3

STEP 11: CREATING QDRANT COLLECTION
Collection name: contract_clauses
✅ Collection created
📐 Vector size: 384
📏 Distance: COSINE

STEP 12: PREPARING VECTORS + PAYLOAD
✅ Prepared 7231 Qdrant points

STEP 13: UPLOADING VECTORS TO QDRANT
Total vectors : 7231
Batch size    : 256
✅ Uploaded 256/7231 (3.5%)
✅ Uploaded 512/7231 (7.1%)
✅ Uploaded 768/7231 (10.6%)
✅ Uploaded 1024/7231 (14.2%)
✅ Uploaded 1280/7231 (17.7%)
✅ Uploaded 1536/7231 (21.2%)
✅ Uploaded 1792/7231 (24.8%)
✅ Uploaded 2048/7231 (28.3%)
✅ Uploaded 2304/7231 (31.9%)
✅ Uploaded 2560/7231 (35.4%)
✅ Uploaded 2816/7231 (38.9%)
✅ Uploaded 3072/723